In [3]:
# Pra mim (becky): conda activate icd_projeto
# conda env export > environment.yml
# conda env create -f environment.yml

# Importações

In [4]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.graph_objects as dict_to_plotly
from plotly.subplots import make_subplots

import openpyxl
import joblib
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# Funções

In [5]:
def region(df):
    """
    Agrupa os estados por regiões
    """
    mapeamento_regioes = {
    # Região Norte
    'AM': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'AC': 'Norte', 'AP': 'Norte', 'TO': 'Norte',
    
    # Região Nordeste
    'MA': 'Nordeste', 'PI': 'Nordeste', 'CE': 'Nordeste', 'RN': 'Nordeste', 'PB': 'Nordeste', 'PE': 'Nordeste', 
    'AL': 'Nordeste', 'SE': 'Nordeste', 'BA': 'Nordeste',
    
    # Região Centro-Oeste
    'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'DF': 'Centro-Oeste',
    
    # Região Sudeste
    'SP': 'Sudeste', 'RJ': 'Sudeste', 'MG': 'Sudeste', 'ES': 'Sudeste',
    
    # Região Sul
    'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul',
    
    # Caso especial do agregado nacional presente na base de dados
    'TOTAL': 'Nacional'
}

    df['region'] = df['state'].map(mapeamento_regioes)

    print("Sucesso: Coluna de regiões criada")
    return df

In [6]:
def classificar_onda(date):
    # Antes da vacinação em massa
    if date < pd.Timestamp('2021-01-17'):
        return '1_pre_vacinacao'
    # Predominância da Gamma (P.1)
    elif date < pd.Timestamp('2021-08-15'):
        return '2_gamma'
    # Predominância da Delta
    elif date < pd.Timestamp('2021-12-15'):
        return '3_delta'
    # Explosão e predominância da Omicron
    elif date < pd.Timestamp('2022-07-01'):
        return '4_omicron'
    # Queda após grandes ondas de transmissão
    else:
        return '5_pos_pico'

In [7]:
def aplicar_medias_moveis(df, colunas_alvo=['newCases', 'newDeaths'], janelas=[7, 15, 30]):
    """
    Faz a média móvel para as colunas de novos casos e mortes nas janelas de 7, 15 e 30 dias
    """
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(by=['state', 'date']).reset_index(drop=True)
    
    for coluna in colunas_alvo:
        for janela in janelas:
            nome_nova_coluna = f"{coluna}_mm{janela}"
            # Aplica o cálculo agrupado por estado
            df[nome_nova_coluna] = df.groupby('state')[coluna].transform(
                lambda x: x.rolling(window=janela, min_periods=1).mean()
            )
            
    print(f"Sucesso: Médias móveis aplicadas para as colunas {colunas_alvo} nas janelas {janelas}.")
    return df

In [8]:
def defasagem(df,colunas_alvo=['newCases'], lags=[7, 14, 21]):
    """
    Faz o cálculo da defasagem na coluna de novos casos para 7, 14 e 21 dias atrás
    """
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(by=['state', 'date']).reset_index(drop=True)

    for coluna in colunas_alvo:
        for lag in lags:
            nome_nova_coluna = f"{coluna}_lag{lag}"
            # O .shift(lag) move os dados daquela coluna 'X' dias para a frente
            df[nome_nova_coluna] = df.groupby('state')[coluna].shift(lag)
            
    print(f"Sucesso: Lags de {lags} dias aplicados para as colunas {colunas_alvo}.")
    return df

In [9]:
# =========================================================
# LAG EPIDEMIOLÓGICO GLOBAL — COVID
# =========================================================

def encontrar_lag_ideal_global(
    df,
    estados=None,
    max_lag=40,
    min_lag=5,
    coluna_casos='newCases',
    coluna_obitos='newDeaths',
    remover_estados_ruidosos=True
):
    """
    Descobre o lag epidemiológico ideal entre CASOS e ÓBITOS.

    Metodologia:
    -------------
    ✓ Cross-correlation temporal
    ✓ Suavização epidemiológica robusta
    ✓ Normalização Z-score
    ✓ Correlação temporal real
    ✓ Resultado individual por estado
    ✓ Resultado médio nacional

    Interpretação:
    --------------
    Mede o atraso médio entre:
        aumento de casos -> aumento de óbitos
    """

    # =====================================================
    # CÓPIA E LIMPEZA
    # =====================================================

    df = df.copy()

    # Remove agregado nacional
    df = df[df['state'] != 'TOTAL']

    # Remove estados muito pequenos/ruidosos
    if remover_estados_ruidosos:

        estados_ruidosos = ['RR', 'AP', 'AC']

        df = df[~df['state'].isin(estados_ruidosos)]

    # Estados analisados
    if estados is None:

        estados = sorted(df['state'].unique())

    # =====================================================
    # ESTRUTURAS
    # =====================================================

    resultados_estados = []

    correlacoes_por_lag = {
        lag: [] for lag in range(max_lag + 1)
    }

    # =====================================================
    # LOOP DOS ESTADOS
    # =====================================================

    for estado in estados:

        df_estado = (
            df[df['state'] == estado]
            .sort_values('date')
            .copy()
        )

        # =================================================
        # SÉRIES TEMPORAIS
        # =================================================

        casos = df_estado[coluna_casos].astype(float)
        obitos = df_estado[coluna_obitos].astype(float)

        # Remove negativos/anomalias
        casos = casos.clip(lower=0)
        obitos = obitos.clip(lower=0)

        # =================================================
        # SUAVIZAÇÃO EPIDEMIOLÓGICA
        # =================================================
        # 14 dias preserva o lag real
        # e reduz ruído administrativo

        casos = casos.rolling(
            window=14,
            center=True
        ).mean()

        obitos = obitos.rolling(
            window=14,
            center=True
        ).mean()

        # =================================================
        # REMOVE NaNs
        # =================================================

        serie = pd.DataFrame({
            'casos': casos,
            'obitos': obitos
        }).dropna()

        casos = serie['casos']
        obitos = serie['obitos']

        # =================================================
        # NORMALIZAÇÃO Z-SCORE
        # =================================================

        casos = (
            (casos - casos.mean())
            / casos.std()
        )

        obitos = (
            (obitos - obitos.mean())
            / obitos.std()
        )

        # =================================================
        # TESTE DOS LAGS
        # =================================================

        correlacoes = []

        for lag in range(max_lag + 1):

            casos_lag = casos.shift(lag)

            temp = pd.DataFrame({
                'obitos': obitos,
                'casos_lag': casos_lag
            }).dropna()

            # Dados insuficientes
            if len(temp) < 30:

                correlacao = np.nan

            else:

                correlacao = temp['obitos'].corr(
                    temp['casos_lag']
                )

            correlacoes.append(correlacao)

            correlacoes_por_lag[lag].append(correlacao)

        # =================================================
        # IGNORA LAGS BIOLOGICAMENTE IMPOSSÍVEIS
        # =================================================

        lags_validos = range(min_lag, max_lag + 1)

        corr_validas = [
            correlacoes[i]
            for i in lags_validos
        ]

        if np.all(np.isnan(corr_validas)):
            continue

        idx_local = np.nanargmax(corr_validas)

        lag_ideal = list(lags_validos)[idx_local]

        corr_ideal = corr_validas[idx_local]

        # =================================================
        # RESULTADO DO ESTADO
        # =================================================

        resultados_estados.append({
            'Estado': estado,
            'Lag Ideal': lag_ideal,
            'Correlação': round(corr_ideal, 4)
        })

    # =====================================================
    # RESULTADOS ESTADUAIS
    # =====================================================

    df_resultados = (
        pd.DataFrame(resultados_estados)
        .sort_values('Lag Ideal')
        .reset_index(drop=True)
    )

    # =====================================================
    # MÉDIA GLOBAL DOS LAGS
    # =====================================================

    medias_globais = []

    for lag in range(max_lag + 1):

        valores = correlacoes_por_lag[lag]

        media = np.nanmean(valores)

        medias_globais.append(media)

    df_media = pd.DataFrame({
        'Lag': range(max_lag + 1),
        'Correlação Média': medias_globais
    })

    # =====================================================
    # MELHOR LAG GLOBAL
    # =====================================================

    df_media_filtrado = df_media[
        df_media['Lag'] >= min_lag
    ]

    idx_global = (
        df_media_filtrado['Correlação Média']
        .idxmax()
    )

    lag_global = int(
        df_media.loc[idx_global, 'Lag']
    )

    corr_global = float(
        df_media.loc[idx_global, 'Correlação Média']
    )

    # =====================================================
    # PRINT DOS RESULTADOS
    # =====================================================

    print("\n============================================")
    print("LAG IDEAL POR ESTADO")
    print("============================================\n")

    print(df_resultados.to_string(index=False))

    print("\n============================================")
    print(f"LAG IDEAL GLOBAL: {lag_global} dias")
    print(f"CORRELAÇÃO MÉDIA: {corr_global:.4f}")
    print("============================================")

    # =====================================================
    # GRÁFICO
    # =====================================================

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df_media['Lag'],
            y=df_media['Correlação Média'],
            mode='lines+markers',
            line=dict(width=3),
            name='Correlação Média'
        )
    )

    # Região epidemiológica esperada
    fig.add_vrect(
        x0=10,
        x1=25,
        fillcolor="green",
        opacity=0.08,
        line_width=0,
        annotation_text="Faixa epidemiológica esperada"
    )

    # Lag ideal
    fig.add_vline(
        x=lag_global,
        line_dash='dash',
        line_color='red',
        annotation_text=f'Lag Ideal = {lag_global} dias'
    )

    fig.update_layout(
        title=(
            '<b>Correlação Cruzada — Casos x Óbitos</b>'
            '<br>Lag Epidemiológico Médio Brasileiro'
        ),
        xaxis_title='Lag (dias)',
        yaxis_title='Correlação Média',
        template='plotly_white',
        width=1000,
        height=550
    )

    fig.show()

    return {
        'lag_global': lag_global,
        'correlacao_global': corr_global,
        'lags_estaduais': df_resultados,
        'media_lags': df_media
    }

# Amostragem

In [10]:
data = pd.read_csv(r"..\data\raw\cases-brazil-states.csv")

In [11]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30842 entries, 0 to 30841
Data columns (total 26 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   epi_week                               30842 non-null  int64  
 1   date                                   30842 non-null  object 
 2   country                                30842 non-null  object 
 3   state                                  30842 non-null  object 
 4   city                                   30842 non-null  object 
 5   newDeaths                              30842 non-null  int64  
 6   deaths                                 30842 non-null  int64  
 7   newCases                               30842 non-null  int64  
 8   totalCases                             30842 non-null  int64  
 9   deathsMS                               30842 non-null  int64  
 10  totalCasesMS                           30842 non-null  int64  
 11  de

In [12]:
# Divisão em uma faixa trimestral pelo conjunto estado-periodo
data['date'] = pd.to_datetime(data['date'])
data['periodo_temporal'] = data['date'].dt.to_period('Q')

In [13]:
# Estrato através do cruzamento entre Estado (state) e Período Temporal
data['estrato'] = data['state'].astype(str) + "_" + data['periodo_temporal'].astype(str)

In [14]:
# Amostragem de 30%
sub = data.groupby('estrato', group_keys=False).sample(frac=0.3, random_state=42)
print(f"Tamanho da subamostra obtida (30%): {sub.shape[0]} linhas.")

Tamanho da subamostra obtida (30%): 9293 linhas.


In [15]:
sub.tail(10)

,epi_week,date,country,state,city,newDeaths,deaths,newCases,totalCases,deathsMS,...,vaccinated,vaccinated_per_100_inhabitants,vaccinated_second,vaccinated_second_per_100_inhabitants,vaccinated_single,vaccinated_single_per_100_inhabitants,vaccinated_third,vaccinated_third_per_100_inhabitants,periodo_temporal,estrato
28824,301,2023-01-05,Brazil,TO,TOTAL,0,4212,0,361001,4212,...,1177095.0,74.83759,1021208.0,64.92657,56219.0,3.5743,572827.0,36.41931,2023Q1,TO_2023Q1
28908,302,2023-01-08,Brazil,TO,TOTAL,0,4212,0,361001,4212,...,1177095.0,74.83759,1021208.0,64.92657,56219.0,3.5743,572827.0,36.41931,2023Q1,TO_2023Q1
30280,309,2023-02-26,Brazil,TO,TOTAL,0,4232,0,365330,4232,...,1181614.0,75.12490,1026612.0,65.27015,56219.0,3.5743,590796.0,37.56175,2023Q1,TO_2023Q1
29048,302,2023-01-13,Brazil,TO,TOTAL,0,4216,0,362364,4216,...,1177095.0,74.83759,1021208.0,64.92657,56219.0,3.5743,572827.0,36.41931,2023Q1,TO_2023Q1
30196,308,2023-02-23,Brazil,TO,TOTAL,0,4232,373,365330,4232,...,1181614.0,75.12490,1026612.0,65.27015,56219.0,3.5743,590796.0,37.56175,2023Q1,TO_2023Q1
30308,309,2023-02-27,Brazil,TO,TOTAL,0,4232,208,365538,4232,...,1181618.0,75.12515,1026630.0,65.27129,56219.0,3.5743,590824.0,37.56353,2023Q1,TO_2023Q1
28852,301,2023-01-06,Brazil,TO,TOTAL,0,4212,0,361001,4212,...,1177095.0,74.83759,1021208.0,64.92657,56219.0,3.5743,572827.0,36.41931,2023Q1,TO_2023Q1
30112,308,2023-02-20,Brazil,TO,TOTAL,0,4232,0,364957,4232,...,1181614.0,75.12490,1026612.0,65.27015,56219.0,3.5743,590796.0,37.56175,2023Q1,TO_2023Q1
29664,305,2023-02-04,Brazil,TO,TOTAL,0,4228,0,364313,4228,...,1180010.0,75.02292,1024583.0,65.14115,56219.0,3.5743,584957.0,37.19052,2023Q1,TO_2023Q1
30532,310,2023-03-07,Brazil,TO,TOTAL,0,4232,0,365538,4232,...,1181618.0,75.12515,1026630.0,65.27129,56219.0,3.5743,590824.0,37.56353,2023Q1,TO_2023Q1


# Tratamento

In [16]:
# Drop das colunas que não agregam informações relevantes: Pais analisado (todos são Brasil), cidade (todas do estado) e a semana da analise
sub = sub.drop(columns=['country', 'city', 'epi_week'])

In [17]:
sub = sub.sort_values(by=['state', 'date']).reset_index(drop=True)

In [18]:
# Verificar duplicadas
sub.duplicated().sum()

np.int64(0)

In [19]:
# Retirar os NaN
sub.fillna(0, inplace=True)

### Criação de Variáveis derivadas

In [20]:
sub = region(sub)

Sucesso: Coluna de regiões criada


In [21]:
sub = aplicar_medias_moveis(sub)

Sucesso: Médias móveis aplicadas para as colunas ['newCases', 'newDeaths'] nas janelas [7, 15, 30].


In [22]:
sub['taxa_letalidade'] = sub['deaths_by_totalCases'] * 100
sub_estados = sub[sub['state'] != 'TOTAL'].copy()

In [23]:
sub['onda'] = sub['date'].apply(classificar_onda)

In [24]:
resultado_lag = encontrar_lag_ideal_global(
    sub,
    max_lag=40,
    min_lag=5
)


LAG IDEAL POR ESTADO

Estado  Lag Ideal  Correlação
    AL          5      0.8319
    AM          5      0.5559
    BA          5      0.8494
    CE          5      0.7395
    ES          5      0.2650
    GO          5      0.2884
    MA          5      0.7001
    MS          5      0.6734
    PI          5      0.8595
    MT          5      0.5081
    PA          5      0.6106
    PB          5      0.5431
    PR          5      0.2375
    PE          5      0.3118
    RN          5      0.3899
    RJ          5      0.0151
    SC          5      0.5829
    SE          5      0.7980
    RO          5      0.6400
    RS          5      0.3902
    SP          5      0.8261
    TO          5      0.6696
    MG          6      0.4091
    DF          6      0.3299

LAG IDEAL GLOBAL: 5 dias
CORRELAÇÃO MÉDIA: 0.5427


In [25]:
for estado in ['SP', 'AM', 'CE', 'RS', 'GO']:
    lag = encontrar_lag_ideal(sub_estados, estado=estado)
    print(f"{estado}: lag ideal = {lag} dias")

NameError: name 'encontrar_lag_ideal' is not defined

In [26]:
sub = defasagem(sub, colunas_alvo=['newCases'], lags=[7, 14, 21])
sub = defasagem(sub, colunas_alvo=['newDeaths'], lags=[7, 14, 21])

Sucesso: Lags de [7, 14, 21] dias aplicados para as colunas ['newCases'].
Sucesso: Lags de [7, 14, 21] dias aplicados para as colunas ['newDeaths'].


In [27]:
sub.shape

(9293, 40)

In [28]:
sub.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9293 entries, 0 to 9292
Data columns (total 40 columns):
 #   Column                                 Non-Null Count  Dtype         
---  ------                                 --------------  -----         
 0   date                                   9293 non-null   datetime64[ns]
 1   state                                  9293 non-null   object        
 2   newDeaths                              9293 non-null   int64         
 3   deaths                                 9293 non-null   int64         
 4   newCases                               9293 non-null   int64         
 5   totalCases                             9293 non-null   int64         
 6   deathsMS                               9293 non-null   int64         
 7   totalCasesMS                           9293 non-null   int64         
 8   deaths_per_100k_inhabitants            9293 non-null   float64       
 9   totalCases_per_100k_inhabitants        9293 non-null   float64 

In [29]:
data = region(data)
data['region'].value_counts(normalize=True)

Sucesso: Coluna de regiões criada


region
Nordeste        0.321023
Norte           0.248719
Sudeste         0.143992
Centro-Oeste    0.142760
Sul             0.107256
Nacional        0.036249
Name: proportion, dtype: float64

In [30]:
sub['region'].value_counts(normalize=True)

region
Nordeste        0.320994
Norte           0.248682
Sudeste         0.143979
Centro-Oeste    0.142796
Sul             0.107285
Nacional        0.036264
Name: proportion, dtype: float64

In [31]:
print(data['date'].min())
print(sub['date'].min())

2020-02-25 00:00:00
2020-02-27 00:00:00


In [32]:
print(data['date'].max())
print(sub['date'].max())

2023-03-18 00:00:00
2023-03-18 00:00:00


# Exploratória

In [33]:
variaveis_foco = [
    'newDeaths',
    'deaths',
    'newCases',
    'totalCases',
    'deathsMS',
    'totalCasesMS',
    'deaths_per_100k_inhabitants',
    'totalCases_per_100k_inhabitants',
    'deaths_by_totalCases',
    'recovered',
    'suspects',
    'tests',
    'tests_per_100k_inhabitants',
    'vaccinated',
    'vaccinated_per_100_inhabitants',
    'vaccinated_second',
    'vaccinated_second_per_100_inhabitants',
    'vaccinated_single',
    'vaccinated_single_per_100_inhabitants',
    'vaccinated_third',
    'vaccinated_third_per_100_inhabitants',
    'newCases_mm7',
    'newCases_mm15',
    'newCases_mm30',
    'newDeaths_mm7',
    'newDeaths_mm15',
    'newDeaths_mm30'
]

sub_estados[variaveis_foco].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
newDeaths,8956.0,23.805,64.365,-43.000,0.000,5.000,20.000,1.282000e+03
deaths,8956.0,16661.815,27179.090,0.000,2462.500,7992.500,18609.250,1.790390e+05
newCases,8956.0,1270.247,2753.740,-850.000,73.000,411.000,1280.000,6.922300e+04
totalCases,8956.0,724636.777,1007811.070,1.000,142619.750,366387.000,855363.500,6.469442e+06
deathsMS,8956.0,16658.708,27179.603,0.000,2461.000,7990.000,18608.750,1.790390e+05
totalCasesMS,8956.0,724477.549,1007850.345,0.000,142530.000,366251.000,854992.000,6.469442e+06
deaths_per_100k_inhabitants,8956.0,207.798,126.720,0.000,92.161,217.658,311.798,4.451330e+02
totalCases_per_100k_inhabitants,8956.0,10687.456,7850.614,0.005,3925.336,9980.215,15226.586,3.293631e+04
deaths_by_totalCases,8956.0,0.024,0.013,0.000,0.016,0.021,0.026,2.110000e-01
recovered,8956.0,612900.912,840638.606,0.000,119470.000,319842.000,699477.000,4.850000e+06


## 1 — Série Temporal: Casos e Óbitos por Estado

In [34]:
def plot_serie_temporal_casos_obitos(df, estados=None, altura=650):
    """
    Série temporal (linhas) de novos casos e novos óbitos por estado,
    usando médias móveis de 7 dias para suavizar ruídos.

    Parâmetros
    ----------
    df      : DataFrame com 'state', 'date', 'newCases_mm7', 'newDeaths_mm7'
    estados : lista de siglas; se None, usa os 5 maiores em casos totais
    altura  : altura do gráfico em pixels
    """
    if estados is None:
        estados = (
            df.groupby('state')['totalCases'].max()
            .nlargest(5).index.tolist()
        )

    df_plot = df[df['state'].isin(estados)].sort_values('date')
    palette = px.colors.qualitative.Plotly

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        subplot_titles=(
            '<b>Novos Casos</b> — Média Móvel 7 dias',
            '<b>Novos Óbitos</b> — Média Móvel 7 dias'
        ),
        vertical_spacing=0.10
    )

    for i, estado in enumerate(estados):
        df_e = df_plot[df_plot['state'] == estado]
        cor  = palette[i % len(palette)]

        fig.add_trace(
            go.Scatter(
                x=df_e['date'], y=df_e['newCases_mm7'],
                name=estado, mode='lines',
                line=dict(color=cor, width=2),
                legendgroup=estado,
            ),
            row=1, col=1
        )
        fig.add_trace(
            go.Scatter(
                x=df_e['date'], y=df_e['newDeaths_mm7'],
                name=estado, mode='lines',
                line=dict(color=cor, width=2, dash='dot'),
                legendgroup=estado,
                showlegend=False,
            ),
            row=2, col=1
        )

    fig.update_layout(
        title='<b>Série Temporal — Novos Casos e Óbitos por Estado</b>',
        title_font_size=18,
        hovermode='x unified',
        template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
        height=altura,
    )
    fig.update_xaxes(title_text='Data', row=2, col=1)
    fig.update_yaxes(title_text='Novos Casos (MM7)',  row=1, col=1)
    fig.update_yaxes(title_text='Novos Óbitos (MM7)', row=2, col=1)

    datas_vacinacao = [
        ("1ª Onda", "2021-01-16", "green"),
        ("2ª Onda", "2021-07-01", "orange"),
        ("3ª Onda", "2021-09-15", "red"),
    ]

    for nome, data, cor in datas_vacinacao:
        fig.add_vline(
            x=data,
            line_width=2,
            line_dash="dash",
            line_color=cor
        )

        fig.add_annotation(
            x=data,
            y=0.98,
            yref="paper",
            text=f"<b>{nome}</b>",
            showarrow=False,
            xanchor="left",
            bgcolor="white",
            bordercolor=cor,
            borderwidth=1,
            font=dict(color=cor, size=11)
        )

    ondas = [
        ("Pré-vacinação", "2020-02-26", "black"),
        ("Gamma",         "2021-01-17", "purple"),
        ("Delta",         "2021-08-15", "blue"),
        ("Ômicron",       "2021-12-15", "red"),
        ("Pós-pico",      "2022-07-01", "green"),
    ]

    for nome, data, cor in ondas:
            fig.add_vline(
                x=data,
                line_color=cor,
                line_dash="dot",
                line_width=2
            )

            fig.add_annotation(
                x=data,
                y=1.06,
                yref="paper",
                text=f"<b>{nome}</b>",
                showarrow=False,
                xanchor="left",
                font=dict(size=10, color=cor)
            )

    fig.show()


plot_serie_temporal_casos_obitos(sub_estados)


## 2 — Série Temporal: Vacinação por Estado

In [35]:
def plot_serie_temporal_vacinacao(df, estados=None, altura=550):
    """
    Série temporal do avanço da vacinação (% da população com 1ª dose)
    por estado, sobreposta à curva de óbitos (MM7) para evidenciar correlação.

    Usa a coluna já existente 'vaccinated_per_100_inhabitants' (porcentagem).

    Parâmetros
    ----------
    df      : DataFrame com 'state', 'date', 'vaccinated_per_100_inhabitants', 'newDeaths_mm7'
    estados : lista de siglas; se None, usa os 5 com maior cobertura vacinal final
    altura  : altura do gráfico em pixels
    """
    if estados is None:
        estados = (
            df.groupby('state')['vaccinated_per_100_inhabitants'].max()
            .nlargest(5).index.tolist()
        )

    df_plot  = df[df['state'].isin(estados)].sort_values('date')
    palette  = px.colors.qualitative.Safe

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    for i, estado in enumerate(estados):
        df_e = df_plot[df_plot['state'] == estado]
        cor  = palette[i % len(palette)]

        # Vacinação — 1ª dose (%) — eixo primário
        fig.add_trace(
            go.Scatter(
                x=df_e['date'], y=df_e['vaccinated_per_100_inhabitants'],
                name=f'{estado} — 1ª Dose (%)',
                mode='lines',
                line=dict(color=cor, width=2),
                legendgroup=estado,
            ),
            secondary_y=False
        )
        # Óbitos MM7 — eixo secundário, tracejado
        fig.add_trace(
            go.Scatter(
                x=df_e['date'], y=df_e['newDeaths_mm7'],
                name=f'{estado} — Óbitos',
                mode='lines',
                line=dict(color=cor, width=1.5, dash='dash'),
                legendgroup=estado,
                showlegend=False,
                opacity=0.6,
            ),
            secondary_y=True
        )

    fig.update_layout(
        title='<b>Série Temporal — Vacinação (1ª Dose %) e Óbitos por Estado</b>',
        title_font_size=18,
        hovermode='x unified',
        template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
        height=altura,
    )
    fig.update_xaxes(title_text='Data')
    fig.update_yaxes(title_text='<b>1ª Dose (% pop.)</b>',    color='SteelBlue', secondary_y=False)
    fig.update_yaxes(title_text='<b>Novos Óbitos (MM7)</b>',  color='Crimson',   secondary_y=True)
    fig.show()


plot_serie_temporal_vacinacao(sub_estados)


## 3 — Mapa Coroplético: Mortalidade e Vacinação por Estado

In [36]:
def plot_mapa_coropletico(df, altura=600):
    """
    Mapa coroplético do Brasil colorindo cada estado pela taxa de óbitos
    por 100 mil habitantes.

    Usa a coluna já existente 'deaths_per_100k_inhabitants'.

    Parâmetros
    ----------
    df    : DataFrame com 'state', 'deaths_per_100k_inhabitants'
    altura: altura do mapa em pixels
    """
    # Snapshot: valor máximo acumulado por estado
    df_mapa = (
        df.groupby('state')['deaths_per_100k_inhabitants']
        .max()
        .reset_index()
    )

    fig = px.choropleth(
        df_mapa,
        geojson='https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson',
        locations='state',
        featureidkey='properties.sigla',
        color='deaths_per_100k_inhabitants',
        color_continuous_scale='Reds',
        range_color=(
            df_mapa['deaths_per_100k_inhabitants'].min(),
            df_mapa['deaths_per_100k_inhabitants'].max()
        ),
        labels={'deaths_per_100k_inhabitants': 'Óbitos / 100k hab.'},
        title='<b>Mortalidade por COVID-19 — Óbitos por 100 mil Habitantes</b>',
    )

    fig.update_geos(fitbounds='locations', visible=False, bgcolor='rgba(0,0,0,0)')
    fig.update_layout(
        template='plotly_white',
        height=altura,
        coloraxis_colorbar=dict(
            title='Óbitos<br>/ 100k hab.',
            thicknessmode='pixels', thickness=18,
            lenmode='fraction', len=0.75,
        ),
        margin=dict(l=0, r=0, t=60, b=0),
    )
    fig.show()


plot_mapa_coropletico(sub_estados)


In [37]:
def plot_mapas_obitos_vacinacao(df, altura=650):
    """
    Compara espacialmente:
    - Óbitos por 100 mil habitantes
    - Cobertura vacinal (% população com 1ª dose)

    usando dois mapas lado a lado.
    """

    # Snapshot final por estado
    df_mapa = (
        df.sort_values('date')
          .groupby('state')
          .last()
          .reset_index()
    )

    geojson_url = (
        "https://raw.githubusercontent.com/"
        "codeforamerica/click_that_hood/master/"
        "public/data/brazil-states.geojson"
    )

    fig = make_subplots(
        rows=1,
        cols=2,
        specs=[[{"type": "choropleth"}, {"type": "choropleth"}]],
        subplot_titles=(
            "Óbitos por 100 mil habitantes",
            "Cobertura Vacinal (1ª dose %)"
        ),
        horizontal_spacing=0.05
    )

    # MAPA 1 — ÓBITOS
    fig.add_trace(
        go.Choropleth(
            geojson=geojson_url,
            locations=df_mapa["state"],
            z=df_mapa["deaths_per_100k_inhabitants"],
            featureidkey="properties.sigla",
            colorscale="Reds",
            colorbar=dict(
                title="Óbitos<br>100k",
                x=0.46
            ),
            marker_line_color="black",
            marker_line_width=0.5,
            name="Óbitos"
        ),
        row=1,
        col=1
    )

    # MAPA 2 — VACINAÇÃO
    fig.add_trace(
        go.Choropleth(
            geojson=geojson_url,
            locations=df_mapa["state"],
            z=df_mapa["vaccinated_per_100_inhabitants"],
            featureidkey="properties.sigla",
            colorscale="Greens",
            colorbar=dict(
                title="Vacinação<br>(%)",
                x=1.02
            ),
            marker_line_color="black",
            marker_line_width=0.5,
            name="Vacinação"
        ),
        row=1,
        col=2
    )

    fig.update_geos(
        fitbounds="locations",
        visible=False,
        scope="south america"
    )

    # Necessário para dois mapas independentes
    fig.update_layout(
        title=(
            "<b>COVID-19 no Brasil</b><br>"
            "Comparação entre Mortalidade e Cobertura Vacinal"
        ),
        template="plotly_white",
        height=altura,
        geo=dict(
            domain={"x": [0.0, 0.45], "y": [0, 1]}
        ),
        geo2=dict(
            domain={"x": [0.55, 1.0], "y": [0, 1]}
        ),
        margin=dict(t=80, l=0, r=0, b=0)
    )

    fig.show()

plot_mapas_obitos_vacinacao(sub)

## 4 — Boxplot: Mortalidade por Região

In [38]:
def plot_boxplot_regiao(df, altura=550):
    """
    Boxplot comparando a distribuição de óbitos por 100 mil habitantes
    entre as cinco regiões geográficas do Brasil.

    Usa a coluna já existente 'deaths_per_100k_inhabitants'.

    Parâmetros
    ----------
    df    : DataFrame com 'region', 'deaths_per_100k_inhabitants'
    altura: altura do gráfico em pixels
    """
    df_plot = df.dropna(subset=['deaths_per_100k_inhabitants', 'region'])

    ordem_regioes = ['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']
    palette = {
        'Norte':        '#1f77b4',
        'Nordeste':     '#ff7f0e',
        'Centro-Oeste': '#2ca02c',
        'Sudeste':      '#d62728',
        'Sul':          '#9467bd',
    }

    fig = go.Figure()
    for regiao in ordem_regioes:
        df_r = df_plot[df_plot['region'] == regiao]
        fig.add_trace(
            go.Box(
                y=df_r['deaths_per_100k_inhabitants'],
                name=regiao,
                boxpoints='outliers',
                marker_color=palette.get(regiao, 'gray'),
                line_color=palette.get(regiao, 'gray'),
                fillcolor=palette.get(regiao, 'gray'),
                opacity=0.7,
            )
        )

    fig.update_layout(
        title='<b>Distribuição de Mortalidade por Região</b><br>Óbitos por 100k hab.',
        title_font_size=18,
        yaxis_title='Óbitos por 100k habitantes',
        xaxis_title='Região',
        template='plotly_white',
        showlegend=False,
        height=altura,
    )
    fig.show()


plot_boxplot_regiao(sub_estados)


## 5 — Scatter Plot: Cobertura Vacinal × Mortalidade

In [39]:
def plot_scatter_vacina_obito(df, altura=600):
    """
    Scatter plot explorando a correlação entre cobertura da 2ª dose (%) e
    óbitos acumulados por 100k hab., com linha de tendência OLS,
    colorido por região e dimensionado pelo total de casos.

    Usa as colunas já existentes:
        'vaccinated_second_per_100_inhabitants' (% 2ª dose)
        'deaths_per_100k_inhabitants'

    Parâmetros
    ----------
    df    : DataFrame — usa snapshot da última data por estado
    altura: altura do gráfico em pixels
    """
    df_snap = (
        df.sort_values('date')
        .groupby('state')
        .last()
        .reset_index()
        .dropna(subset=['vaccinated_second_per_100_inhabitants', 'deaths_per_100k_inhabitants'])
    )

    # Remove zeros que distorceriam a correlação (estados sem dado de vacina)
    df_snap = df_snap[
        (df_snap['vaccinated_second_per_100_inhabitants'] > 0) &
        (df_snap['deaths_per_100k_inhabitants'] > 0)
    ]

    r, p = stats.pearsonr(
        df_snap['vaccinated_second_per_100_inhabitants'],
        df_snap['deaths_per_100k_inhabitants']
    )

    # Linha de tendência manual com numpy (sem statsmodels)
    x_vals = df_snap['vaccinated_second_per_100_inhabitants'].values
    y_vals = df_snap['deaths_per_100k_inhabitants'].values
    coef   = np.polyfit(x_vals, y_vals, 1)
    x_line = np.linspace(x_vals.min(), x_vals.max(), 200)
    y_line = np.polyval(coef, x_line)

    fig = px.scatter(
        df_snap,
        x='vaccinated_second_per_100_inhabitants',
        y='deaths_per_100k_inhabitants',
        color='region',
        size='totalCases',
        text='state',
        labels={
            'vaccinated_second_per_100_inhabitants': '2ª Dose (% da população)',
            'deaths_per_100k_inhabitants':           'Óbitos por 100k hab.',
            'region':                                'Região',
        },
        title=(
            f'<b>Cobertura Vacinal (2ª Dose) × Mortalidade por Estado</b>'
            f'<br>r de Pearson = {r:.3f}  |  p-valor = {p:.4f}'
        ),
        color_discrete_sequence=px.colors.qualitative.Safe,
        template='plotly_white',
        height=altura,
    )
    fig.update_traces(
        textposition='top center',
        selector=dict(mode='markers+text')
    )
    # Adiciona a reta de tendência manualmente
    fig.add_trace(
        go.Scatter(
            x=x_line, y=y_line,
            mode='lines',
            name=f'Tendência (y = {coef[0]:.2f}x + {coef[1]:.1f})',
            line=dict(color='black', width=2, dash='dash'),
            showlegend=True,
        )
    )
    fig.update_layout(
        title_font_size=17,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    )
    fig.show()


plot_scatter_vacina_obito(sub_estados)


In [40]:
# %% [markdown]
# ## 8 — Scatter Temporal com Lag: Vacinação × Óbitos Fututos

# %%
def plot_scatter_lag_vacina_obitos(
    df,
    lag_dias=21,
    altura=650
):
    """
    Relaciona cobertura vacinal atual com óbitos futuros (lag epidemiológico),
    permitindo investigar se maior vacinação está associada
    à redução posterior da mortalidade.

    Estratégia:
    ----------
    vacinação(t)  --->  óbitos(t + lag)

    Usa:
        vaccinated_second_per_100_inhabitants
        newDeaths_mm7

    Parâmetros
    ----------
    df        : DataFrame principal
    lag_dias  : defasagem temporal entre vacinação e óbitos
    altura    : altura do gráfico
    """

    df_plot = df.copy()

    # =====================================================
    # ORDENAÇÃO TEMPORAL
    # =====================================================

    df_plot = (
        df_plot
        .sort_values(['state', 'date'])
        .reset_index(drop=True)
    )

    # =====================================================
    # CRIA ÓBITOS FUTUROS
    # =====================================================

    df_plot[f'obitos_futuros_{lag_dias}d'] = (
        df_plot
        .groupby('state')['newDeaths_mm7']
        .shift(-lag_dias)
    )

    # =====================================================
    # REMOVE VALORES INVÁLIDOS
    # =====================================================

    df_plot = df_plot[
        (df_plot['vaccinated_second_per_100_inhabitants'] > 0)
    ].dropna(
        subset=[
            'vaccinated_second_per_100_inhabitants',
            f'obitos_futuros_{lag_dias}d'
        ]
    )

    # =====================================================
    # AGREGA POR ESTADO
    # =====================================================

    df_estado = (
        df_plot
        .groupby(['state', 'region'])
        .agg({
            'vaccinated_second_per_100_inhabitants': 'mean',
            f'obitos_futuros_{lag_dias}d': 'mean',
            'totalCases': 'max'
        })
        .reset_index()
    )

    # =====================================================
    # CORRELAÇÃO
    # =====================================================

    r, p = stats.pearsonr(
        df_estado['vaccinated_second_per_100_inhabitants'],
        df_estado[f'obitos_futuros_{lag_dias}d']
    )

    # =====================================================
    # LINHA DE TENDÊNCIA
    # =====================================================

    x_vals = df_estado['vaccinated_second_per_100_inhabitants'].values
    y_vals = df_estado[f'obitos_futuros_{lag_dias}d'].values

    coef = np.polyfit(x_vals, y_vals, 1)

    x_line = np.linspace(x_vals.min(), x_vals.max(), 200)
    y_line = np.polyval(coef, x_line)

    # =====================================================
    # SCATTER
    # =====================================================

    fig = px.scatter(
        df_estado,
        x='vaccinated_second_per_100_inhabitants',
        y=f'obitos_futuros_{lag_dias}d',
        color='region',
        size='totalCases',
        text='state',
        labels={
            'vaccinated_second_per_100_inhabitants':
                'Cobertura Vacinal — 2ª Dose (%)',

            f'obitos_futuros_{lag_dias}d':
                f'Óbitos MM7 {lag_dias} dias depois',

            'region': 'Região'
        },
        title=(
            f'<b>Vacinação × Óbitos Futuros ({lag_dias} dias)</b>'
            f'<br>Correlação temporal com lag epidemiológico'
            f'<br>r = {r:.3f} | p-valor = {p:.4f}'
        ),
        template='plotly_white',
        color_discrete_sequence=px.colors.qualitative.Safe,
        height=altura
    )

    fig.update_traces(
        textposition='top center'
    )

    # =====================================================
    # RETA DE TENDÊNCIA
    # =====================================================

    fig.add_trace(
        go.Scatter(
            x=x_line,
            y=y_line,
            mode='lines',
            name=f'Tendência (y = {coef[0]:.2f}x + {coef[1]:.2f})',
            line=dict(
                color='black',
                dash='dash',
                width=2
            )
        )
    )

    # =====================================================
    # AJUSTES VISUAIS
    # =====================================================

    fig.update_layout(
        title_font_size=18,
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=1.02,
            xanchor='center',
            x=0.5
        )
    )

    fig.show()


# =========================================================
# EXECUÇÃO
# =========================================================

plot_scatter_lag_vacina_obitos(
    sub_estados,
    lag_dias=21
)

## 6 — Heatmap de Correlação: Todas as Variáveis Numéricas

In [41]:
def plot_heatmap_correlacao(df, altura=800):
    """
    Heatmap da matriz de correlação de Pearson entre todas as variáveis
    numéricas, útil para detectar multicolinearidade antes da modelagem.
    Exibe apenas o triângulo inferior para evitar redundância.

    Parâmetros
    ----------
    df    : DataFrame completo (sub_estados)
    altura: altura do gráfico em pixels
    """
    # Exclui colunas puramente identificadoras ou derivadas de índice
    colunas_excluir = ['epi_week']
    numericas = df.select_dtypes(include='number').drop(
        columns=[c for c in colunas_excluir if c in df.columns],
        errors='ignore'
    )

    corr = numericas.corr(method='pearson').round(2)

    # Máscara triangular superior
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    corr_lower = corr.where(~mask)

    fig = go.Figure(
        go.Heatmap(
            z=corr_lower.values,
            x=corr_lower.columns.tolist(),
            y=corr_lower.index.tolist(),
            colorscale='RdBu_r',
            zmid=0, zmin=-1, zmax=1,
            text=corr_lower.round(2).values,
            texttemplate='%{text}',
            textfont=dict(size=8),
            colorbar=dict(title='r de<br>Pearson'),
            hoverongaps=False,
        )
    )
    fig.update_layout(
        title='<b>Heatmap de Correlação — Variáveis Numéricas</b>',
        title_font_size=18,
        template='plotly_white',
        height=altura,
        xaxis=dict(tickangle=-45),
        margin=dict(l=180, b=180),
    )
    fig.show()


plot_heatmap_correlacao(sub_estados)


## 7 — Gráfico de Barras: Ranking de Letalidade por Estado

In [42]:
def plot_barras_letalidade(df, altura=620):
    """
    Gráfico de barras horizontais com o ranking de taxa de letalidade
    por estado, ordenado do maior para o menor.

    Usa a coluna derivada 'taxa_letalidade' (deaths_by_totalCases × 100),
    calculada na célula de preparação.

    Parâmetros
    ----------
    df    : DataFrame com 'state', 'taxa_letalidade', 'region'
    altura: altura do gráfico em pixels
    """
    # Snapshot: última data disponível por estado
    df_snap = (
        df.sort_values('date')
        .groupby('state')
        .last()
        .reset_index()
        .dropna(subset=['taxa_letalidade'])
        .sort_values('taxa_letalidade', ascending=True)  # crescente → barras ordenadas
    )

    mapa_cores = {
        'Norte':        '#1f77b4',
        'Nordeste':     '#ff7f0e',
        'Centro-Oeste': '#2ca02c',
        'Sudeste':      '#d62728',
        'Sul':          '#9467bd',
    }
    cores = df_snap['region'].map(mapa_cores).fillna('#aec7e8').tolist()

    fig = go.Figure(
        go.Bar(
            x=df_snap['taxa_letalidade'],
            y=df_snap['state'],
            orientation='h',
            marker_color=cores,
            text=df_snap['taxa_letalidade'].map('{:.2f}%'.format),
            textposition='outside',
            hovertemplate='%{y}: %{x:.2f}%<extra></extra>',
        )
    )

    media = df_snap['taxa_letalidade'].mean()
    fig.add_vline(
        x=media, line_dash='dash', line_color='black',
        annotation_text=f'Média: {media:.2f}%',
        annotation_position='top right',
    )

    # Legenda manual por região
    for regiao, cor in mapa_cores.items():
        fig.add_trace(
            go.Bar(x=[None], y=[None], marker_color=cor, name=regiao, showlegend=True)
        )

    fig.update_layout(
        title='<b>Ranking de Letalidade por Estado</b><br>Taxa = (Óbitos Totais / Casos Totais) × 100',
        title_font_size=18,
        xaxis_title='Taxa de Letalidade (%)',
        yaxis_title='Estado',
        template='plotly_white',
        height=altura,
        margin=dict(r=90),
        barmode='overlay',
        legend=dict(
            title='Região',
            orientation='v',
            yanchor='bottom', y=0.01,
            xanchor='right',  x=0.99,
        ),
    )
    fig.show()


plot_barras_letalidade(sub_estados)


## 8 — Histograma da Variável Resposta: deaths_per_100k_inhabitants

In [43]:
def plot_histograma_variavel_resposta(df, altura=620):
    """
    Histograma de 'deaths_per_100k_inhabitants' em escala original e
    após transformação log1p, dispostos lado a lado.

    Exibe também estatísticas descritivas (média, mediana, assimetria)
    e uma curva KDE sobreposta para avaliar a forma da distribuição.
    A comparação entre os dois painéis serve de base para decidir se
    a transformação log1p será aplicada à variável resposta no EP3.

    Parâmetros
    ----------
    df    : DataFrame com coluna 'deaths_per_100k_inhabitants' (sub_estados)
    altura: altura do gráfico em pixels
    """
    coluna = 'deaths_per_100k_inhabitants'
    serie = df[coluna].dropna()
    serie_log = np.log1p(serie)

    def stats_label(s, nome):
        return (
            f"<b>{nome}</b><br>"
            f"Média: {s.mean():.2f}<br>"
            f"Mediana: {s.median():.2f}<br>"
            f"Assimetria: {s.skew():.2f}<br>"
            f"Curtose: {s.kurt():.2f}"
        )

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            '<b>Escala Original</b>',
            '<b>Após Transformação log1p</b>'
        ),
        horizontal_spacing=0.12
    )

    cor_original = '#636EFA'
    cor_log      = '#EF553B'

    for col_idx, (s, cor, rotulo) in enumerate(
        [
            (serie,     cor_original, 'Original'),
            (serie_log, cor_log,      'log1p')
        ],
        start=1
    ):
        # Histograma
        fig.add_trace(
            go.Histogram(
                x=s,
                nbinsx=60,
                name=rotulo,
                marker_color=cor,
                opacity=0.75,
                histnorm='probability density',
                showlegend=False,
            ),
            row=1, col=col_idx
        )

        # Curva KDE via numpy
        kde_x = np.linspace(s.min(), s.max(), 300)
        kernel = stats.gaussian_kde(s, bw_method='scott')
        kde_y  = kernel(kde_x)

        fig.add_trace(
            go.Scatter(
                x=kde_x,
                y=kde_y,
                mode='lines',
                name=f'KDE — {rotulo}',
                line=dict(color=cor, width=3),
                showlegend=True,
            ),
            row=1, col=col_idx
        )

        # Linha vertical: média
        fig.add_vline(
            x=s.mean(),
            line_dash='dash',
            line_color='black',
            line_width=1.5,
            annotation_text=f'Média={s.mean():.1f}',
            annotation_position='top right',
            row=1, col=col_idx  # type: ignore[call-arg]
        )

        # Linha vertical: mediana
        fig.add_vline(
            x=s.median(),
            line_dash='dot',
            line_color='gray',
            line_width=1.5,
            annotation_text=f'Mediana={s.median():.1f}',
            annotation_position='top left',
            row=1, col=col_idx  # type: ignore[call-arg]
        )

        # Anotação com estatísticas no canto
        fig.add_annotation(
            xref='x domain' if col_idx == 1 else 'x2 domain',
            yref='y domain' if col_idx == 1 else 'y2 domain',
            x=0.98, y=0.97,
            text=stats_label(s, rotulo),
            showarrow=False,
            align='right',
            bgcolor='rgba(255,255,255,0.85)',
            bordercolor='lightgray',
            borderwidth=1,
            font=dict(size=11),
            row=1, col=col_idx
        )

    fig.update_layout(
        title=(
            '<b>Distribuição da Variável Resposta — '
            'deaths_per_100k_inhabitants</b>'
            '<br>Comparação: escala original vs. transformação log1p'
        ),
        title_font_size=17,
        template='plotly_white',
        height=altura,
        legend=dict(
            orientation='h',
            yanchor='bottom', y=1.04,
            xanchor='center', x=0.5
        ),
        bargap=0.02,
    )

    fig.update_xaxes(title_text='Óbitos por 100k hab.', row=1, col=1)
    fig.update_xaxes(title_text='log1p(Óbitos por 100k hab.)', row=1, col=2)
    fig.update_yaxes(title_text='Densidade', row=1, col=1)

    fig.show()


plot_histograma_variavel_resposta(sub_estados)


## 9 — Análise por Onda Epidemiológica

In [ ]:
def plot_analise_por_onda(df, altura=800):
    """
    Análise comparativa da variável 'onda' em três dimensões:

    Painel 1 (linha): evolução temporal da mediana de novos óbitos
    (MM7) por onda e região, permitindo comparar a gravidade de
    cada período entre as regiões do Brasil.

    Painel 2 (boxplot): distribuição de deaths_per_100k por onda,
    revelando dispersão e outliers em cada período epidemiológico.

    Painel 3 (barras empilhadas): cobertura vacinal média da 1ª dose
    por onda e região, evidenciando o nível de imunização ao longo
    das fases da pandemia.

    Parâmetros
    ----------
    df    : DataFrame com 'onda', 'region', 'date',
            'newDeaths_mm7', 'deaths_per_100k_inhabitants',
            'vaccinated_per_100_inhabitants'  (sub_estados)
    altura: altura total do gráfico em pixels
    """

    if 'onda' not in df.columns:
        df = df.copy()
        df['onda'] = df['date'].apply(classificar_onda)

    # Ordem e rótulos legíveis
    ordem_ondas = [
        '1_pre_vacinacao',
        '2_gamma',
        '3_delta',
        '4_omicron',
        '5_pos_pico',
    ]
    rotulos_ondas = {
        '1_pre_vacinacao': 'Pré-vacinação',
        '2_gamma':         'Gamma',
        '3_delta':         'Delta',
        '4_omicron':       'Ômicron',
        '5_pos_pico':      'Pós-pico',
    }
    df = df.copy()
    df['onda_label'] = df['onda'].map(rotulos_ondas)

    ordem_labels = [rotulos_ondas[o] for o in ordem_ondas]
    regioes       = ['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']

    mapa_cores_regiao = {
        'Norte':        '#1f77b4',
        'Nordeste':     '#ff7f0e',
        'Centro-Oeste': '#2ca02c',
        'Sudeste':      '#d62728',
        'Sul':          '#9467bd',
    }

    # Paleta para as ondas (boxplot / barras)
    cores_ondas = px.colors.qualitative.Safe

    # =====================================================
    # SUBPLOTS
    # =====================================================
    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=(
            '<b>Mediana de Novos Óbitos (MM7) por Onda e Região</b>',
            '<b>Distribuição de Novos Óbitos Diários (MM7) por Onda Epidemiológica</b>',
            '<b>Cobertura Vacinal Média (1ª Dose %) por Onda e Região</b>',
        ),
        vertical_spacing=0.10,
        row_heights=[0.36, 0.34, 0.30],
    )

    # =====================================================
    # PAINEL 1 — Linha: mediana de newDeaths_mm7 por onda × região
    # =====================================================
    df_p1 = (
        df.dropna(subset=['newDeaths_mm7', 'onda_label', 'region'])
        .groupby(['onda_label', 'region'], observed=True)['newDeaths_mm7']
        .median()
        .reset_index()
    )
    df_p1['onda_label'] = pd.Categorical(
        df_p1['onda_label'], categories=ordem_labels, ordered=True
    )
    df_p1 = df_p1.sort_values('onda_label')

    for regiao in regioes:
        sub_r = df_p1[df_p1['region'] == regiao]
        fig.add_trace(
            go.Scatter(
                x=sub_r['onda_label'],
                y=sub_r['newDeaths_mm7'],
                mode='lines+markers',
                name=regiao,
                legendgroup=regiao,
                showlegend=True,
                line=dict(color=mapa_cores_regiao[regiao], width=2.5),
                marker=dict(size=8),
            ),
            row=1, col=1
        )

    # =====================================================
    # PAINEL 2 — Boxplot: deaths_per_100k por onda
    # =====================================================
    for i, onda_label in enumerate(ordem_labels):
        sub_o = df[df['onda_label'] == onda_label]['newDeaths_mm7'].dropna()
        fig.add_trace(
            go.Box(
                y=sub_o,
                name=onda_label,
                legendgroup=onda_label,
                showlegend=False,
                marker_color=cores_ondas[i % len(cores_ondas)],
                boxmean='sd',
                line_width=1.5,
            ),
            row=2, col=1
        )

    # =====================================================
    # PAINEL 3 — Barras agrupadas: vacinação média por onda × região
    # =====================================================
    df_p3 = (
        df.dropna(subset=['vaccinated_per_100_inhabitants', 'onda_label', 'region'])
        .groupby(['onda_label', 'region'], observed=True)['vaccinated_per_100_inhabitants']
        .mean()
        .reset_index()
    )
    df_p3['onda_label'] = pd.Categorical(
        df_p3['onda_label'], categories=ordem_labels, ordered=True
    )
    df_p3 = df_p3.sort_values('onda_label')

    for regiao in regioes:
        sub_r = df_p3[df_p3['region'] == regiao]
        fig.add_trace(
            go.Bar(
                x=sub_r['onda_label'],
                y=sub_r['vaccinated_per_100_inhabitants'],
                name=regiao,
                legendgroup=regiao,
                showlegend=False,
                marker_color=mapa_cores_regiao[regiao],
                opacity=0.85,
            ),
            row=3, col=1
        )

    # =====================================================
    # AJUSTES VISUAIS
    # =====================================================
    fig.update_layout(
        title=(
            '<b>Análise Comparativa por Onda Epidemiológica</b>'
            '<br>Mortalidade, distribuição de óbitos e vacinação '
            'em cada fase da pandemia'
        ),
        title_font_size=17,
        template='plotly_white',
        height=altura,
        barmode='group',
        legend=dict(
            title='Região',
            orientation='v',
            yanchor='middle', y=0.85,
            xanchor='left',   x=1.01,
        ),
        margin=dict(r=130),
    )

    fig.update_yaxes(title_text='Mediana Óbitos MM7',       row=1, col=1)
    fig.update_yaxes(title_text='Novos Óbitos Diários MM7', row=2, col=1)
    fig.update_yaxes(title_text='1ª Dose (% pop.)',          row=3, col=1)

    fig.show()

plot_analise_por_onda(sub)

## 10 — Limiar de Vacinação e Mortalidade

In [53]:
def plot_limiar_vacinacao(df, largura_bin=5, coluna_obitos='newDeaths_mm7',
                          coluna_vacina='vaccinated_second_per_100_inhabitants',
                          altura=750):
    """
    Investiga o limiar de cobertura vacinal (2ª dose) a partir do qual
    a mortalidade cai de forma significativa — questão (a) do estudo.
 
    Metodologia
    -----------
    1. Filtra apenas o período pós-vacinação (vacina > 0).
    2. Divide a cobertura vacinal em bins de largura fixa (padrão: 5 pp).
    3. Calcula mediana e IQR de newDeaths_mm7 por bin.
    4. Detecta o limiar como o bin de maior queda relativa
       (Δ% negativo entre medianas consecutivas).
    5. Apresenta três painéis complementares:
       - Painel 1 (curva + IC): mediana suavizada por bin, com faixa
         IQR e linha vertical marcando o limiar detectado.
       - Painel 2 (barras): contagem de observações por bin —
         revela onde há dados suficientes para confiar na estimativa.
       - Painel 3 (scatter por região): scatter vacinação × óbitos
         colorido por região, com linha de limiar, para verificar se
         o limiar é consistente entre regiões ou apenas nacional.
 
    Parâmetros
    ----------
    df             : DataFrame (sub_estados)
    largura_bin    : largura de cada intervalo de cobertura vacinal (pp)
    coluna_obitos  : variável resposta (padrão: newDeaths_mm7)
    coluna_vacina  : cobertura vacinal (padrão: vaccinated_second_per_100)
    altura         : altura total do gráfico em pixels
    """
 
    # =====================================================
    # PREPARAÇÃO
    # =====================================================
    df = df.copy()
    df = df[df[coluna_vacina] > 0].dropna(subset=[coluna_vacina, coluna_obitos])
    df = df[df[coluna_obitos] >= 0]
 
    # Bins de vacinação com passo fixo
    vacc_max  = int(np.ceil(df[coluna_vacina].max() / largura_bin) * largura_bin)
    bins      = np.arange(0, vacc_max + largura_bin, largura_bin)
    midpoints = bins[:-1] + largura_bin / 2
 
    df['vacc_bin']  = pd.cut(df[coluna_vacina], bins=bins, labels=midpoints, right=False)
    df['vacc_bin']  = df['vacc_bin'].astype(float)
 
    # =====================================================
    # ESTATÍSTICAS POR BIN
    # =====================================================
    resumo = (
        df.groupby('vacc_bin', observed=True)[coluna_obitos]
        .agg(
            mediana='median',
            q25=lambda x: x.quantile(0.25),
            q75=lambda x: x.quantile(0.75),
            n='count'
        )
        .reset_index()
        .sort_values('vacc_bin')
    )
 
    # Remove bins com menos de 10 obs (ruído)
    resumo = resumo[resumo['n'] >= 10].reset_index(drop=True)
 
    # =====================================================
    # DETECÇÃO DO LIMIAR
    # =====================================================
    # Maior queda relativa entre medianas consecutivas
    resumo['delta_rel'] = resumo['mediana'].pct_change()          # Δ%
    idx_limiar  = resumo['delta_rel'].idxmin()                    # bin da maior queda
    limiar_vacc = float(resumo.loc[idx_limiar, 'vacc_bin'])
    limiar_med  = float(resumo.loc[idx_limiar, 'mediana'])
 
    queda_pct   = abs(resumo.loc[idx_limiar, 'delta_rel']) * 100  # % de queda
 
    # =====================================================
    # CORES
    # =====================================================
    # =====================================================
    # FIGURA
    # =====================================================
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=(
            '<b>Mediana de Novos Óbitos (MM7) por Cobertura Vacinal (2ª Dose)</b>',
            '<b>Número de Observações por Bin — confiabilidade da estimativa</b>',
        ),
        vertical_spacing=0.12,
        row_heights=[0.72, 0.28],
    )
 
    # =========================================================
    # PAINEL 1 — Curva de mediana + faixa IQR + limiar
    # =========================================================
 
    # Faixa IQR
    fig.add_trace(
        go.Scatter(
            x=list(resumo['vacc_bin']) + list(resumo['vacc_bin'])[::-1],
            y=list(resumo['q75'])      + list(resumo['q25'])[::-1],
            fill='toself',
            fillcolor='rgba(99, 110, 250, 0.15)',
            line=dict(color='rgba(0,0,0,0)'),
            name='IQR (25%–75%)',
            showlegend=True,
        ),
        row=1, col=1
    )
 
    # Curva de mediana
    fig.add_trace(
        go.Scatter(
            x=resumo['vacc_bin'],
            y=resumo['mediana'],
            mode='lines+markers',
            name='Mediana de novos óbitos MM7',
            line=dict(color='#636EFA', width=3),
            marker=dict(size=7, color='#636EFA'),
        ),
        row=1, col=1
    )
 
    # Anotação do ponto de limiar
    fig.add_trace(
        go.Scatter(
            x=[limiar_vacc],
            y=[limiar_med],
            mode='markers',
            name=f'Limiar detectado: {limiar_vacc:.0f}%',
            marker=dict(
                size=14, color='crimson',
                symbol='diamond',
                line=dict(width=2, color='white')
            ),
        ),
        row=1, col=1
    )
 
    # Linha vertical do limiar — painel 1
    fig.add_vline(
        x=limiar_vacc,
        line_dash='dash',
        line_color='crimson',
        line_width=2,
        annotation_text=(
            f'  Limiar: {limiar_vacc:.0f}% 2ª dose<br>'
            f'  Queda de {queda_pct:.1f}% na mediana'
        ),
        annotation_position='top right',
        annotation_font_color='crimson',
        row=1, col=1
    )
 
    # =========================================================
    # PAINEL 2 — Contagem de obs por bin
    # =========================================================
    fig.add_trace(
        go.Bar(
            x=resumo['vacc_bin'],
            y=resumo['n'],
            name='Observações por bin',
            marker_color=[
                'crimson' if v == limiar_vacc else 'rgba(99,110,250,0.5)'
                for v in resumo['vacc_bin']
            ],
            showlegend=False,
        ),
        row=2, col=1
    )
    fig.add_vline(
        x=limiar_vacc,
        line_dash='dash',
        line_color='crimson',
        line_width=1.5,
        row=2, col=1
    )
 
    # =========================================================
    # LAYOUT
    # =========================================================
    fig.update_layout(
        title=f'<b>Limiar de Vacinação e Mortalidade por COVID-19</b>',
        title_font_size=17,
        template='plotly_white',
        height=altura,
        legend=dict(
            orientation='v',
            yanchor='top',    y=0.99,
            xanchor='right',  x=0.99,
            bgcolor='rgba(255,255,255,0.85)',
            bordercolor='lightgray',
            borderwidth=1,
            font=dict(size=12),
        ),
        margin=dict(t=60, r=40),
        bargap=0.1,
        annotations=[
            dict(
                text=(
                    f'Maior queda detectada em ~{limiar_vacc:.0f}% de 2ª dose '
                    f'({queda_pct:.1f}% de redução na mediana)'
                ),
                xref='paper', yref='paper',
                x=0, y=1.04,
                xanchor='left', yanchor='bottom',
                showarrow=False,
                font=dict(size=12, color='gray'),
            )
        ],
    )
 
    fig.update_xaxes(title_text='2ª Dose (% da população)', row=1, col=1)
    fig.update_xaxes(title_text='2ª Dose (% da população)', row=2, col=1)
 
    fig.update_yaxes(title_text='Mediana Novos Óbitos MM7', row=1, col=1)
    fig.update_yaxes(title_text='Nº observações',           row=2, col=1)
 
    fig.show()
 
    # =========================================================
    # PRINT DO RESULTADO
    # =========================================================
    print("\n" + "=" * 52)
    print("LIMIAR DE VACINAÇÃO — RESULTADO")
    print("=" * 52)
    print(f"  Variável vacinal : {coluna_vacina}")
    print(f"  Variável resposta: {coluna_obitos}")
    print(f"  Largura do bin   : {largura_bin} pp")
    print(f"\n  Limiar detectado : {limiar_vacc:.0f}% de 2ª dose")
    print(f"  Queda na mediana : {queda_pct:.1f}% em relação ao bin anterior")
    print(f"  Mediana pré-limiar : "
          f"{float(resumo.loc[idx_limiar - 1, 'mediana']) if idx_limiar > 0 else 'N/A':.1f} óbitos MM7")
    print(f"  Mediana pós-limiar : {limiar_med:.1f} óbitos MM7")
    print("=" * 52 + "\n")
 
    return {
        'limiar_vacc_pct': limiar_vacc,
        'queda_pct':       queda_pct,
        'resumo_bins':     resumo,
    }
 
 
resultado_limiar = plot_limiar_vacinacao(sub_estados)


LIMIAR DE VACINAÇÃO — RESULTADO
  Variável vacinal : vaccinated_second_per_100_inhabitants
  Variável resposta: newDeaths_mm7
  Largura do bin   : 5 pp

  Limiar detectado : 18% de 2ª dose
  Queda na mediana : 59.3% em relação ao bin anterior
  Mediana pré-limiar : 30.6 óbitos MM7
  Mediana pós-limiar : 12.4 óbitos MM7

